# Experiment 12 — Variable plaque and episodic disease activity

**Question.** Under regular forcing the model loses attachment steadily once a site converts, whereas clinical progression is site-specific and episodic. Does variability in plaque from one recall interval to the next make disease activity episodic, and does memory change the pattern?

**Setting.** Comparator backbone with structural memory (Exp 09: $\gamma=0.8$, $h_D=0.8$, $\delta=0.001$). Plaque regrows as in Exp 07 ($k=0.05$, $T=60$), but the peak reached in each interval is drawn at random from a log-normal distribution with mean $\bar p$ and coefficient of variation CV. The same random peak sequences are used for every host.

**Two maintenance regimes.**
* **A, no therapy:** plaque is removed at every recall, nothing else. This tests whether random peaks alone produce remissions.
* **B, treat on detection:** at every recall, a site that is inflamed at the visit ($H>0.5$) is also debrided ($M\to0$). This is closer to supportive periodontal therapy.

**Hosts.** Instant adaptation; slow memory $\varepsilon=0.03$ and $0.01$.

**Read-outs.** Activity is defined per interval as attachment loss above $10^{-4}$. Read-outs: onset time; fraction of active intervals; clustering of activity (lag-1 autocorrelation, run lengths); and burstiness, the share of total attachment loss that falls in the 10% most destructive intervals (about 0.10–0.15 when loss is spread evenly, approaching 1 when it is concentrated in a few bursts).

**Sections**
* **A** no therapy: onset times and remissions
* **B** treat on detection: clustering and burstiness
* **C** checks, summary and export

In [ ]:
# =====================================================================
# EXPERIMENT 12 — folder setup
# =====================================================================
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/Biosemiotic_Perio")
except ImportError:                      # local execution
    BASE_DIR = Path.cwd() / "Biosemiotic_Perio"
EXP_DIR = BASE_DIR / "Experiment_12_Episodic_Activity"
EXP_DIR.mkdir(parents=True, exist_ok=True)
QUICK = os.environ.get("EXP_QUICK") == "1"     # set True for a fast smoke test with reduced grids
print(f"Experiment folder:\n{EXP_DIR}\nExists: {EXP_DIR.exists()}   QUICK={QUICK}")

In [ ]:
import sys, json, time, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import brentq

versions_text = f"""
Experiment: 12 Variable plaque and episodic activity
Created: {datetime.now(timezone.utc).isoformat()}

Python: {sys.version}
Platform: {platform.platform()}
NumPy: {np.__version__}
Pandas: {pd.__version__}
SciPy: {scipy.__version__}
Matplotlib: {matplotlib.__version__}
"""
with open(EXP_DIR / "software_versions.txt", "w") as f:
    f.write(versions_text)
print(versions_text)

# ---- figure style (same tokens as Experiment 07) ------------------------
INK, INK2, GRIDC, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
MEM_COL = {0.3: "#86b6ef", 0.1: "#5598e7", 0.03: "#2a78d6", 0.01: "#1c5cab", 0.003: "#0d366b"}
INSTANT_COL = "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 1.6,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "legend.frameon": False,
})
def save_fig(fig, name):
    fig.savefig(EXP_DIR / f"{name}.jpg", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =====================================================================
# CORE MODEL — state (H, M, A, D), A = ln(theta0/theta), D = structural memory
# =====================================================================
#   dH/dt = -H + F(M + u(t); theta0*exp(-A))
#   dM/dt =  r*(-M + G(H) + gamma*D)              pocket niche supports the community
#   dA/dt =  eps*(alpha*H - A)                     response-state memory ('adaptive')
#   dD/dt =  delta*max(H - hD, 0)*(1 - D)          irreversible, saturating tissue loss
# mode 'adaptive' (slow memory) | 'instant' (A = alpha*H: same equilibria, no memory
# timescale) | 'frozen' (A held fixed).  Fixed-step RK4, vectorised over elements.
COMPARATOR = dict(n=4.0, q=4.0, K=0.9, th0=0.8, r=1.0, a=1.0)   # Exp 06B backbone
DT = 0.1
U_UP, U_DOWN = 0.4682392215, 0.0630204768                       # static folds (D = 0)


def hill(x, h, n):
    x = np.maximum(x, 0.0)
    xn = x ** n
    return xn / (xn + h ** n)


def rhs(H, M, A, D, u, eps, p, mode, gamma, delta, hD):
    A_eff = p['a'] * H if mode == 'instant' else A
    dH = -H + hill(M + u, p['th0'] * np.exp(-A_eff), p['n'])
    dM = p['r'] * (-M + hill(H, p['K'], p['q']) + gamma * D)
    dA = eps * (p['a'] * H - A) if mode == 'adaptive' else 0.0 * A
    dD = delta * np.maximum(H - hD, 0.0) * (1.0 - D)
    return dH, dM, dA, dD


def run(y0, u_fn, n_steps, eps=0.0, mode='adaptive', p=COMPARATOR, gamma=0.0,
        delta=0.0, hD=0.8, events=None, recorder=None, dt=DT):
    """u_fn(i, c): forcing for step i at stage offset c in {0, .5, 1};
    events(i, H, M, A, D) -> state, applied at the start of step i;
    recorder(i, H, M, A, D), called after each step."""
    H, M, A, D = [np.array(v, float, copy=True) for v in np.broadcast_arrays(*y0)]
    eps, gamma, delta = [np.broadcast_to(np.asarray(v, float), H.shape) for v in (eps, gamma, delta)]
    if isinstance(p.get('r'), np.ndarray):
        p = dict(p, r=np.broadcast_to(p['r'], H.shape))
    f = lambda h, m, a, d, u: rhs(h, m, a, d, u, eps, p, mode, gamma, delta, hD)
    for i in range(n_steps):
        if events is not None:
            H, M, A, D = events(i, H, M, A, D)
        u0, uh, u1 = u_fn(i, 0.0), u_fn(i, 0.5), u_fn(i, 1.0)
        k1 = f(H, M, A, D, u0)
        k2 = f(H + .5*dt*k1[0], M + .5*dt*k1[1], A + .5*dt*k1[2], D + .5*dt*k1[3], uh)
        k3 = f(H + .5*dt*k2[0], M + .5*dt*k2[1], A + .5*dt*k2[2], D + .5*dt*k2[3], uh)
        k4 = f(H + dt*k3[0], M + dt*k3[1], A + dt*k3[2], D + dt*k3[3], u1)
        H = H + dt/6 * (k1[0] + 2*k2[0] + 2*k3[0] + k4[0])
        M = M + dt/6 * (k1[1] + 2*k2[1] + 2*k3[1] + k4[1])
        A = A + dt/6 * (k1[2] + 2*k2[2] + 2*k3[2] + k4[2])
        D = D + dt/6 * (k1[3] + 2*k2[3] + 2*k3[3] + k4[3])
        if recorder is not None:
            recorder(i, H, M, A, D)
    if mode == 'instant':
        A = p['a'] * H
    return H, M, A, D


def piecewise_u(boundaries, values):
    """Piecewise-constant forcing: segment k+1 starts at step boundaries[k] (per element)."""
    B = np.stack([np.asarray(b, np.int64) for b in boundaries])
    V = np.stack([np.asarray(v, float) for v in values])
    def u_fn(i, c):
        seg = (i >= B).sum(axis=0)
        return np.take_along_axis(V, seg[None, :], 0)[0]
    return u_fn


def u_branch(H, p=COMPARATOR, gamma=0.0, D=0.0):
    """Equilibrium forcing as a function of H* (M1 with structural memory D frozen)."""
    th = p['th0'] * np.exp(-p['a'] * H)
    return th * (H / (1 - H)) ** (1 / p['n']) - hill(H, p['K'], p['q']) - gamma * D


def low_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = np.linspace(1e-9, 0.6, 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i], Hs[i + 1], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H


def high_equilibrium(u, p=COMPARATOR, gamma=0.0, D=0.0):
    Hs = 1 - np.logspace(-12, np.log10(0.5), 20001)
    f = u_branch(Hs, p, gamma, D) - u
    i = np.where(np.sign(f[:-1]) != np.sign(f[1:]))[0][0]
    H = brentq(lambda h: u_branch(h, p, gamma, D) - u, Hs[i + 1], Hs[i], xtol=1e-15)
    return H, hill(H, p['K'], p['q']) + gamma * D, p['a'] * H

In [ ]:
# =====================================================================
# PARAMETERS
# =====================================================================
T, K_REG = 60.0, 0.05
GAMMA, DELTA, H_D = 0.8, 0.001, 0.8
HOSTS = [('instant', 0.0), ('adaptive', 0.03), ('adaptive', 0.01)]
MEANS = np.array([0.45, 0.50, 0.55, 0.60])
CVS = np.array([0.15, 0.30])
REPS, N_CYC, SEED = 100, 150, 20261015
ACTIVE = 1e-4
if QUICK:
    MEANS = np.array([0.50, 0.55]); CVS = np.array([0.30]); REPS, N_CYC = 10, 40

Ns = int(round(T / DT)); F_PK = 1 - np.exp(-K_REG * T)
host_name = lambda mode, eps: 'instant' if mode == 'instant' else f'eps={eps}'
HCOL = {'instant': INSTANT_COL, 'eps=0.03': MEM_COL[0.03], 'eps=0.01': MEM_COL[0.01]}

# one random peak sequence per (mean, CV, replicate), shared by all hosts
mm, cc, rr = np.meshgrid(MEANS, CVS, np.arange(REPS), indexing='ij'); mm, cc, rr = mm.ravel(), cc.ravel(), rr.ravel()
sig = np.sqrt(np.log(1 + cc ** 2)); mu = np.log(mm) - sig ** 2 / 2
rng = np.random.default_rng(SEED)
PEAKS = np.exp(mu[:, None] + sig[:, None] * rng.standard_normal((mm.size, N_CYC)))
UMAX = PEAKS / F_PK
print(f"{mm.size} sites x {N_CYC} intervals; mean realized peak by condition:")
display(pd.DataFrame(dict(mean=mm, cv=cc, realized=PEAKS.mean(1))).groupby(['mean', 'cv']).realized.mean().round(3))

def simulate(mode, eps, treat_on_detection):
    n = mm.size
    Hmin = np.zeros((n, N_CYC)); Hvisit = np.zeros((n, N_CYC)); D_end = np.zeros((n, N_CYC)); debrided = np.zeros((n, N_CYC), bool)
    cur = {}
    def u_fn(i, c):
        cyc = min(i // Ns, N_CYC - 1)
        return UMAX[:, cyc] * (1 - np.exp(-K_REG * (((i % Ns) + c) * DT)))
    def ev(i, H, M, A, D):
        if treat_on_detection and i > 0 and i % Ns == 0:
            fl = H > 0.5; debrided[:, i // Ns] = fl; M = np.where(fl, 0.0, M)
        return H, M, A, D
    def rec(i, H, M, A, D):
        if i % Ns == 0: cur['m'] = H.copy()
        cur['m'] = np.minimum(cur['m'], H)
        if (i + 1) % Ns == 0:
            c = (i + 1) // Ns - 1; Hmin[:, c] = cur['m']; Hvisit[:, c] = H; D_end[:, c] = D
    run((np.zeros(n), 0.0, 0.0, 0.0), u_fn, N_CYC * Ns, eps=eps, mode=mode, gamma=GAMMA, delta=DELTA, hD=H_D,
        events=ev, recorder=rec)
    dD = np.diff(np.c_[np.zeros(n), D_end], axis=1)
    return dict(Hmin=Hmin, Hvisit=Hvisit, D=D_end, dD=dD, debrided=debrided)

def runs(row):
    """lengths of runs of True and of False in a boolean sequence"""
    on, off, r, last = [], [], 0, None
    for v in row:
        if v == last: r += 1
        else:
            if last is not None: (on if last else off).append(r)
            r, last = 1, v
    if last is not None: (on if last else off).append(r)
    return on, off

def metrics(sim, sl):
    act = sim['dD'][sl] > ACTIVE
    tot = sim['dD'][sl].sum(1)
    k = max(1, int(round(0.1 * N_CYC)))
    top = np.sort(sim['dD'][sl], 1)[:, -k:].sum(1)
    share = np.where(tot > 1e-6, top / np.maximum(tot, 1e-12), np.nan)
    ac = [np.corrcoef(r[:-1], r[1:])[0, 1] for r in act.astype(float) if 0 < r.mean() < 1 and r[:-1].std() > 0 and r[1:].std() > 0]
    on_all, off_all = [], []
    for row in act:
        on, off = runs(row); on_all += on; off_all += off
    return dict(active_fraction=act.mean(), lag1_autocorr=np.mean(ac) if ac else np.nan,
                mean_active_run=np.mean(on_all) if on_all else np.nan, mean_quiet_run=np.mean(off_all) if off_all else np.nan,
                top10_share=np.nanmean(share), D_final=sim['D'][sl, -1].mean())
print("ready")

## A. No therapy: onset times and remissions

Each site starts naive. *Onset* is the first interval in which the site is entrapped ($H>0.5$ throughout). A *remission* is a later interval in which it is not.

In [ ]:
t0 = time.time(); simA = {}; rowsA = []
for mode, eps in HOSTS:
    h = host_name(mode, eps); simA[h] = s = simulate(mode, eps, False)
    ent = s['Hmin'] > 0.5
    onset = np.where(ent.any(1), ent.argmax(1), np.nan)
    remiss = np.array([(~ent[k, int(onset[k]):]).sum() if np.isfinite(onset[k]) else 0 for k in range(mm.size)])
    for m in MEANS:
        for cv in CVS:
            sl = (mm == m) & (cc == cv); o = onset[sl]
            rowsA.append(dict(host=h, mean_peak=m, cv=cv, converted=np.isfinite(o).mean(),
                              onset_median=np.nanmedian(o), onset_q25=np.nanpercentile(o, 25) if np.isfinite(o).any() else np.nan,
                              onset_q75=np.nanpercentile(o, 75) if np.isfinite(o).any() else np.nan,
                              remission_intervals_per_site=remiss[sl].mean(), D_final=s['D'][sl, -1].mean()))
    print(h, f"{time.time() - t0:.0f}s")
dfA = pd.DataFrame(rowsA); dfA.to_csv(EXP_DIR / "no_therapy_onset.csv", index=False); display(dfA.round(3))

fig, ax = plt.subplots(1, len(CVS), figsize=(5 * len(CVS), 3.6), constrained_layout=True, squeeze=False); ax = ax[0]
for j, cv in enumerate(CVS):
    for h, s in simA.items():
        for m, ls in zip(MEANS, ['-', '--', ':', '-.']):
            sl = (mm == m) & (cc == cv); ent = s['Hmin'][sl] > 0.5
            first = np.where(ent.any(1), ent.argmax(1), N_CYC + 1)
            surv = [(first > c).mean() for c in range(N_CYC)]
            ax[j].plot(np.arange(N_CYC), surv, ls, color=HCOL[h], lw=1.1, label=f"{h}, mean {m:g}" if j == 0 else None)
    ax[j].set(xlabel="maintenance interval (T = 60)", ylabel="fraction of sites not yet converted", title=f"No therapy, CV = {cv:g}", ylim=(-0.02, 1.02))
ax[0].legend(fontsize=6, ncol=2)
save_fig(fig, "fig12_01_onset_no_therapy")

## B. Treat on detection

At every recall, a site inflamed at the visit is debrided. Activity can now switch on and off.

In [ ]:
t0 = time.time(); simB = {}; rowsB = []
for mode, eps in HOSTS:
    h = host_name(mode, eps); simB[h] = s = simulate(mode, eps, True)
    for m in MEANS:
        for cv in CVS:
            sl = (mm == m) & (cc == cv)
            met = metrics(s, sl)
            half = N_CYC // 3
            early = metrics(dict(dD=s['dD'][:, :half], D=s['D'][:, :half]), sl)
            late = metrics(dict(dD=s['dD'][:, -half:], D=s['D'][:, -half:]), sl)
            rowsB.append(dict(host=h, mean_peak=m, cv=cv, **met, debridements_per_site=s['debrided'][sl].sum(1).mean(),
                              quiet_run_first_third=early['mean_quiet_run'], quiet_run_last_third=late['mean_quiet_run']))
    print(h, f"{time.time() - t0:.0f}s")
dfB = pd.DataFrame(rowsB); dfB.to_csv(EXP_DIR / "treat_on_detection_metrics.csv", index=False); display(dfB.round(3))

# example sites: same random peak sequences, instant vs slow memory, near threshold
m_ex = MEANS[1] if len(MEANS) > 1 else MEANS[0]; cv_ex = CVS[-1]
idx = np.where((mm == m_ex) & (cc == cv_ex))[0][:4]
fig, ax = plt.subplots(len(simB), 1, figsize=(10, 1.9 * len(simB)), constrained_layout=True, sharex=True)
for a_, (h, s) in zip(ax, simB.items()):
    for k, j in enumerate(idx):
        a_.plot(np.arange(N_CYC), s['D'][j], color=HCOL[h], lw=1.0, alpha=0.5 + 0.12 * k)
        act = np.where(s['dD'][j] > ACTIVE)[0]; a_.plot(act, np.full(act.size, -0.04 - 0.03 * k), '|', color=HCOL[h], ms=4)
    a_.set(ylabel="attachment loss D", title=f"{h}: four sites, mean peak {m_ex:g}, CV {cv_ex:g} (ticks: active intervals)")
ax[-1].set_xlabel("maintenance interval")
save_fig(fig, "fig12_02_example_sites")

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for h in simB:
    for cv, ls in zip(CVS, ['--', '-']):
        g = dfB[(dfB.host == h) & (dfB.cv == cv)]
        ax[0].plot(g.mean_peak, g.top10_share, 'o' + ls, color=HCOL[h], label=f"{h}, CV {cv:g}")
        ax[1].plot(g.mean_peak, g.lag1_autocorr, 'o' + ls, color=HCOL[h])
        ax[2].plot(g.mean_peak, g.D_final, 'o' + ls, color=HCOL[h])
ax[0].axhline(0.1, color=INK2, lw=0.7, ls=':')
ax[0].set(xlabel="mean peak plaque", ylabel="share of loss in top 10% of intervals", title="Burstiness"); ax[0].legend(fontsize=6.5)
ax[1].set(xlabel="mean peak plaque", ylabel="lag-1 autocorrelation of activity", title="Clustering")
ax[2].set(xlabel="mean peak plaque", ylabel=f"attachment loss after {N_CYC} intervals", title="Cumulative loss")
save_fig(fig, "fig12_03_burstiness")

## C. Checks, summary and export

In [ ]:
checks = []
checks.append(dict(check="D stays within [0, 1]", value=bool(all((s['D'] >= -1e-12).all() and (s['D'] <= 1 + 1e-9).all() for s in list(simA.values()) + list(simB.values())))))
checks.append(dict(check="D never decreases", value=bool(all((s['dD'] >= -1e-12).all() for s in list(simA.values()) + list(simB.values())))))
for h in simA:
    g = dfA[dfA.host == h].sort_values(['cv', 'mean_peak'])
    mono = all(np.all(np.diff(gg.converted.values) >= -0.05) for _, gg in g.groupby('cv'))
    checks.append(dict(check=f"{h}: fraction converted non-decreasing in mean peak (no therapy, tolerance 0.05)", value=bool(mono)))
dfChk = pd.DataFrame(checks); dfChk.to_csv(EXP_DIR / "checks.csv", index=False); display(dfChk)
summary = pd.concat([
    dfA.assign(item=lambda d: "no therapy: remission intervals per site, " + d.host + f", mean " + d.mean_peak.astype(str) + ", CV " + d.cv.astype(str)).rename(columns={'remission_intervals_per_site': 'value'})[['item', 'value']],
    dfB.assign(item=lambda d: "treat on detection: top-10% loss share, " + d.host + ", mean " + d.mean_peak.astype(str) + ", CV " + d.cv.astype(str)).rename(columns={'top10_share': 'value'})[['item', 'value']],
    dfB.assign(item=lambda d: "treat on detection: lag-1 autocorrelation, " + d.host + ", mean " + d.mean_peak.astype(str) + ", CV " + d.cv.astype(str)).rename(columns={'lag1_autocorr': 'value'})[['item', 'value']],
])
summary.to_csv(EXP_DIR / "summary.csv", index=False)
with open(EXP_DIR / "parameters.json", "w") as f:
    json.dump(dict(backbone=COMPARATOR, T=T, k=K_REG, gamma=GAMMA, delta=DELTA, h_D=H_D, hosts=[host_name(*h) for h in HOSTS],
                   means=MEANS.tolist(), cvs=CVS.tolist(), reps=REPS, n_intervals=N_CYC, seed=SEED, active_threshold=ACTIVE, dt=DT), f, indent=2)
np.savez_compressed(EXP_DIR / "peak_sequences.npz", peaks=PEAKS, mean=mm, cv=cc)
print("saved")

In [ ]:
readme = """Experiment 12 - Variable plaque and episodic disease activity
Comparator backbone with structural memory (gamma=0.8, h_D=0.8, delta=0.001). Plaque regrows (k=0.05) and is removed
every T=60; the peak of each interval is log-normal (mean, CV), same sequences for every host.
A: no therapy (plaque removal only). B: treat on detection (debridement M->0 at a recall if H>0.5 at the visit).
Hosts: instant; slow memory eps=0.03, 0.01. Active interval: attachment loss > 1e-4.
Files
- no_therapy_onset.csv, fig12_01_onset_no_therapy          onset times, conversion fraction, remissions
- treat_on_detection_metrics.csv, fig12_02_example_sites, fig12_03_burstiness   activity, clustering, burstiness, loss
- peak_sequences.npz, checks.csv, summary.csv, parameters.json
"""
with open(EXP_DIR / "README.txt", "w") as f:
    f.write(readme)
archive = shutil.make_archive(str(BASE_DIR / "Experiment_12_Episodic_Activity_EXPORT"), 'zip', EXP_DIR)
print("Saved:", archive)